In [ ]:
import sqlite3
import shutil
from urllib.parse import urlparse
import os
import sys
import pandas as pd
from bs4 import BeautifulSoup as bs
from bs4.formatter import XMLFormatter

## 一、从Chrome缓存中提取网站的 favicon 图标

In [55]:
def get_domain_from_url(url):
    """
    从任意URL中提取域名
    支持：http/https、带端口、带路径、带参数的所有URL
    """
    if not url.startswith(("http://", "https://")):
        url = "https://" + url  # 补全协议，否则解析失败
    
    parsed = urlparse(url)
    domain = parsed.netloc.split(":")[0]  # 去掉端口号
    return domain

# 测试示例
if __name__ == "__main__":
    # 各种格式都能正确解析
    test_urls = [
        "https://www.baidu.com/index.html",
        "https://blog.github.com:8080/test?a=1",
        "https://mail.163.com",
        "www.bilibili.com",  # 不带协议也能解析
        "https://www.google.co.uk"
    ]
    
    for url in test_urls:
        domain = get_domain_from_url(url)
        print(f"URL: {url}\n域名: {domain}\n")

URL: https://www.baidu.com/index.html
域名: www.baidu.com

URL: https://blog.github.com:8080/test?a=1
域名: blog.github.com

URL: https://mail.163.com
域名: mail.163.com

URL: www.bilibili.com
域名: www.bilibili.com

URL: https://www.google.co.uk
域名: www.google.co.uk



In [56]:
# 图标路径（是一个sqlite文件）
favicons_path = os.path.join(
    os.getenv('LOCALAPPDATA'),
    r'Google\Chrome\User Data\Default\Favicons'
)

# 网址与icon_id的映射关系
sql = """
select t.icon_id,
       t.page_url
from icon_mapping t
"""
with sqlite3.connect(favicons_path) as conn:
    cur = conn.cursor()
    cur.execute(sql)
    row = cur.fetchall()

In [57]:
# 从网址中提取域名
df = pd.DataFrame(row, columns=['icon_id', 'url'])
df['domain_name'] = df.url.apply(get_domain_from_url)
df_domain_name = df.groupby(['icon_id', 'domain_name']).count().reset_index()
df_domain_name.head()

,icon_id,domain_name,url
0,1,cn.bing.com,2
1,2,www.chromium.org,1
2,3,support.google.com,2
3,4,chrome.google.com,1
4,5,xiaolai.li,1


In [63]:
# 提取favicon图标数据
sql = """
select t.icon_id,
       t.width,
       t.height,
       t.image_data
from favicon_bitmaps t
"""
with sqlite3.connect(favicons_path) as conn:
    cur = conn.cursor()
    cur.execute(sql)
    row = cur.fetchall()

df_img = pd.DataFrame(row, columns=['icon_id','width','height','image_data'])

In [64]:
# 存储到 `./output-favicons/domain.png` 文件
for i in df_img.index: 
    print(i, end='\r')
    icon_id, data = df_img.loc[i,['icon_id','image_data']]
    domain_name_list = df_domain_name.loc[df_domain_name.icon_id==icon_id, 'domain_name'].to_list()
    for d in domain_name_list:
        with open(f'./output-favicons/{d}.png', 'wb') as f:
            f.write(data)

## 二、从【典藏】中的 Markdown 中解析网站域名，并保存相关图片到 Select 文件夹，更新Markdown

In [ ]:
# 🔥 关键：自定义 XML 不排序格式化器
class UnsortedXMLFormatter(XMLFormatter):
    def attributes(self, tag):
        # 直接按原始顺序输出，绝不排序
        yield from tag.attrs.items()

In [194]:
# 解析 XML 要带父标签
html="""
<CardGrid>
<LinkCard title="OI Wiki" href="https://oi-wiki.org/" icon="mdi:web" description="信息学奥林匹克竞赛"/>
<LinkCard title="IMO" href="https://www.imo-official.org/" icon="mdi:web" description="国际数学奥林匹克"/>
<LinkCard title="Using your Head is Permitted" href="https://www.brand.site.co.il/riddles/usingyourhead.html" icon="mdi:web" description=""/>
<LinkCard title="Project Euler" href="https://projecteuler.net/" icon="mdi:web" description=""/>
<LinkCard title="欧拉计划" href="https://pe-cn.github.io/" icon="mdi:web" description=""/>
<LinkCard title="LeetCode/Problems" href="https://leetcode.com/problemset/" icon="mdi:web" description=""/>
<LinkCard title="小木虫" href="https://muchong.com/" icon="mdi:web" description=""/>
</CardGrid>
"""

new_html = bs(html, 'xml')
lc_list = new_html.select('LinkCard')
lc_list

[<LinkCard description="信息学奥林匹克竞赛" href="https://oi-wiki.org/" icon="mdi:web" title="OI Wiki"/>,
 <LinkCard description="国际数学奥林匹克" href="https://www.imo-official.org/" icon="mdi:web" title="IMO"/>,
 <LinkCard description="" href="https://www.brand.site.co.il/riddles/usingyourhead.html" icon="mdi:web" title="Using your Head is Permitted"/>,
 <LinkCard description="" href="https://projecteuler.net/" icon="mdi:web" title="Project Euler"/>,
 <LinkCard description="" href="https://pe-cn.github.io/" icon="mdi:web" title="欧拉计划"/>,
 <LinkCard description="" href="https://leetcode.com/problemset/" icon="mdi:web" title="LeetCode/Problems"/>,
 <LinkCard description="" href="https://muchong.com/" icon="mdi:web" title="小木虫"/>]

In [195]:
target_dir = './select/'

for lc in lc_list: 
    domain = get_domain_from_url(lc.get('href'))
    source = f'./output-favicons/{domain}.png' 
    if os.path.exists(source):
        shutil.copy(source, target_dir)
        lc["icon"] = f"/web-favicons/{domain}.png"

In [196]:
# 输出新HTML
# 其中 encode(formatter=UnsortedXMLFormatter()).decode() 是为了保持原来标签中每个属性的顺序
print('\n'.join([str(_.encode(formatter=UnsortedXMLFormatter()).decode()) for _ in lc_list]))

<LinkCard title="OI Wiki" href="https://oi-wiki.org/" icon="/web-favicons/oi-wiki.org.png" description="信息学奥林匹克竞赛"/>
<LinkCard title="IMO" href="https://www.imo-official.org/" icon="/web-favicons/www.imo-official.org.png" description="国际数学奥林匹克"/>
<LinkCard title="Using your Head is Permitted" href="https://www.brand.site.co.il/riddles/usingyourhead.html" icon="mdi:web" description=""/>
<LinkCard title="Project Euler" href="https://projecteuler.net/" icon="/web-favicons/projecteuler.net.png" description=""/>
<LinkCard title="欧拉计划" href="https://pe-cn.github.io/" icon="/web-favicons/pe-cn.github.io.png" description=""/>
<LinkCard title="LeetCode/Problems" href="https://leetcode.com/problemset/" icon="/web-favicons/leetcode.com.png" description=""/>
<LinkCard title="小木虫" href="https://muchong.com/" icon="/web-favicons/muchong.com.png" description=""/>
